# Gaussian: direct and indirect PMP versus analytical gold

This notebook only loads saved results and calls plotting helpers. Training, test-bank generation, posterior sampling, log marginal likelihoods, calibration bins, and error diagnostics run in the Python command-line workflow.

The candidates are always **m1, m2, m3, m4**, with prior probability **1/4** each. Models **m5–m12** are test-only OOD sources. All methods and summary dimensions use the same persisted raw test observations. **D** is the raw observation dimension, **N** is the number of observations, and **S** is the learned summary dimension.

Set `GAUSSIAN_PMP_RESULTS` to a completed comparison directory and `GAUSSIAN_DIRECT_RUN` to its direct training directory. The defaults below use D=20, N=10, S=20. Run this notebook with the kernel working directory inside the project.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

GAUSSIAN_ROOT = next(
    parent / "benchmark" / "examples" / "gaussian"
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "benchmark" / "examples" / "gaussian" / "config.py").is_file()
)
sys.path.insert(0, str(GAUSSIAN_ROOT.parents[2]))
from benchmark.examples.gaussian.direct.pmp_plots import (
    load_saved_results, plot_loss, plot_calibration, plot_pmp_recovery, plot_paired_tv,
)

RESULTS = Path(os.environ.get(
    "GAUSSIAN_PMP_RESULTS",
    GAUSSIAN_ROOT / "results/direct_pmp/D20_N10_S20/comparison",
))
RUN = Path(os.environ.get(
    "GAUSSIAN_DIRECT_RUN",
    GAUSSIAN_ROOT / "networks/direct_pmp/D20_N10_S20",
))
history, probabilities, paired, calibration = load_saved_results(RESULTS, RUN)

## Training and independent validation

Checkpoint selection uses validation loss. Final test results are not used for checkpoint selection.

In [ ]:
plot_loss(history)
plt.show()

## Independent ID test calibration

The saved one-versus-rest reliability bins pool only m1–m4 test sources under the equal model prior. Empty bins are omitted from the plot. OOD sources are excluded from this calibration assessment.

In [ ]:
plot_calibration(calibration, probabilities)
plt.show()

## Recovery of analytical gold model probabilities

Each point is a matched dataset and candidate model. Circles denote ID sources; crosses denote OOD sources. Both methods are plotted against the same analytical gold probabilities.

In [ ]:
plot_pmp_recovery(probabilities, "direct")
plt.show()
plot_pmp_recovery(probabilities, "indirect")
plt.show()

## Paired total variation errors

The saved errors are TV = 0.5 × Σ |p̂ − p_gold| and ΔTV = TV_direct − TV_indirect. Negative ΔTV means direct is closer to gold. Every matched dataset is retained, including difficult samples with low indirect ESS. Configuration identifiers appear in the figure titles.

In [ ]:
plot_paired_tv(paired)
plt.show()